# AI Tabanlı Çöp Sınıflandırma Sistemi

**Proje Amacı:** Uçtan uca bir derin öğrenme projesi geliştirerek, geri dönüşüm kutularına atılan nesneleri (Plastik, Cam, Kağıt, Metal vb.) kapalı ve siyah bir hazne içerisinde tespit eden sistemin yapay zeka modelini eğitmek.

### Adım 1: Kütüphane Kurulumları ve Ortam Optimizasyonu

**Dependency Hell:** Kurulum aşamasında Colab'in varsayılan numpy sürümü ile TensorFlow ve rembg arasında bağımlılık çakışmaları yaşandı. Ayrıca Colab Pro GPU'sunda yüklü olan cupy kütüphanesi, rembg'nin arka plan silme işlemi sırasında bellek hatalarına yol açtı.

Bu sorunu çözmek için numpy ve scipy sürümlerini belirli versiyonlara sabitledik ve uyumsuzluk yaratan cupy kütüphanesini sistemden kaldırarak işlemlerin standart donanım hızlandırıcılarıyla stabil çalışmasını sağladık.

In [6]:
# 1. Kütüphane Kurulumları (Versiyonları TF ile uyumlu kalmalı)
!pip install -q "numpy<2" "scipy<1.13" rembg kagglehub onnxruntime

# Rembg'nin CPU/Standart Numpy ile çalışması için uyumsuz CuPy'i kaldırıyoruz
!pip uninstall -y cupy-cuda12x cupy

# 2. İçe Aktarmalar
import os
import shutil
import numpy as np
import tensorflow as tf
import kagglehub
from google.colab import drive

# 3. Drive Bağlantısı
drive.mount('/content/drive')

print("Kurulum tamamlandı! TensorFlow Sürümü:", tf.__version__)
print("Numpy Sürümü:", np.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Kurulum tamamlandı! TensorFlow Sürümü: 2.20.0
✅ Numpy Sürümü: 1.26.4


### Adım 2: Veri Setlerinin İçe Aktarılması

Daha yüksek bir doğruluk hedefine ulaşmak için sadece kendi topladığımız yerel verilerle sınırlı kalmadık. Kaggle üzerinden 3 farklı geniş kapsamlı çöp veri setini kagglehub kullanarak doğrudan Colab belleklerine indirdik. Orijinal verilerimiz ise Google Drive üzerinden sisteme entegre edildi.

In [7]:
# Kaggle Veri Setlerini İndir
print("Kaggle veri setleri indiriliyor...")
kaggle1_path = kagglehub.dataset_download("karansolanki01/garbage-classification")
kaggle2_path = kagglehub.dataset_download("hassnainzaidi/garbage-classification")
kaggle3_path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")

# Drive verisetinin kısayol klasörünün yolu
drive_dataset_path = "/content/drive/MyDrive/Geri_Donusum_Veri_Seti"

print("Tüm veri yolları tanımlandı!")

Kaggle veri setleri indiriliyor...
Using Colab cache for faster access to the 'garbage-classification' dataset.
Using Colab cache for faster access to the 'garbage-classification' dataset.
Using Colab cache for faster access to the 'garbage-classification' dataset.
✅ Tüm veri yolları tanımlandı!


### Adım 3: Siyah Hazne Simülasyonu ve Akıllı Veri Havuzu

Şartname gereği model, içi tamamen siyah ve LED aydınlatmalı bir makine haznesinde çalışacaktır. Dış kaynaklı fotoğrafların arka planları bu kuralı ihlal ediyordu.

Colab ortamında zaman kaybını önlemek için öncelikle klasörün dolu olup olmadığı kontrol edilir. Eğer veriler daha önce işlenmemişse; rembg kullanılarak dış kaynaklı tüm verilerin arka planları kod üzerinden piksel piksel silinir ve tamamen siyah (0, 0, 0, 255) bir zeminine oturtulur. Maksimum 1500 görsel kotasıyla dengeli bir master_dataset oluşturulur.

In [8]:
import os
import glob
from PIL import Image
from rembg import remove

MASTER_DIR = '/content/master_dataset'
CLASSES = ['plastic', 'glass', 'paper', 'metal', 'trash', 'cardboard']
MAX_IMAGES_PER_CLASS = 1500 # Eğitimi optimize etmek için kota
# Hızlı bir eğitim için sayı düşürülebilir fakat accuracy ciddi şekilde düşer
# Örnek: Parametre 200'e ayarlandığı zaman accuracy değeri 0.63 oldu.

# 1. AKILLI KONTROL (Zaman tasarrufu için)
is_ready = False
if os.path.exists(MASTER_DIR):
    # Klasördeki toplam dosya sayısını hesapla
    total_files = sum([len(files) for r, d, files in os.walk(MASTER_DIR)])
    if total_files > 50: # İçeride veri varsa işlemi atla
        is_ready = True

if is_ready:
    print("'master_dataset' zaten dolu! Uzun süren arka plan silme işlemi atlanıyor...")
else:
    print("'master_dataset' hazırlanıyor. Bu işlem zaman alabilir...")

    # Sınıf klasörlerini oluştur
    for c in CLASSES:
        os.makedirs(os.path.join(MASTER_DIR, c), exist_ok=True)

    def process_and_black_bg(img_path, save_path, size=(224, 224)):
        try:
            input_img = Image.open(img_path).convert("RGBA")
            output_img = remove(input_img)
            black_bg = Image.new("RGBA", output_img.size, (0, 0, 0, 255))
            black_bg.paste(output_img, (0, 0), output_img)
            final_img = black_bg.convert("RGB").resize(size)
            final_img.save(save_path, "JPEG")
            return True
        except: return False

    def process_standard(img_path, save_path, size=(224, 224)):
        try:
            img = Image.open(img_path).convert("RGB").resize(size)
            img.save(save_path, "JPEG")
            return True
        except: return False

    kaggle_paths = [kaggle1_path, kaggle2_path, kaggle3_path]

    print("Kaggle verileri işleniyor...")
    for k_path in kaggle_paths:
        for root, dirs, files in os.walk(k_path):
            folder_name = os.path.basename(root).lower()
            matched_class = next((c for c in CLASSES if c in folder_name), None)

            if matched_class:
                img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                current_count = len(os.listdir(os.path.join(MASTER_DIR, matched_class)))

                for img_file in img_files:
                    if current_count >= MAX_IMAGES_PER_CLASS: break
                    full_img_path = os.path.join(root, img_file)
                    save_full_path = os.path.join(MASTER_DIR, matched_class, f"kaggle_{current_count}.jpg")
                    if process_and_black_bg(full_img_path, save_full_path): current_count += 1

    print("Drive'daki kendi verileriniz işleniyor...")
    for root, dirs, files in os.walk(drive_dataset_path):
        folder_name = os.path.basename(root).lower()
        matched_class = next((c for c in CLASSES if c in folder_name), None)

        if matched_class:
            img_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            current_count = len(os.listdir(os.path.join(MASTER_DIR, matched_class)))
            for img_file in img_files:
                full_img_path = os.path.join(root, img_file)
                save_full_path = os.path.join(MASTER_DIR, matched_class, f"drive_{current_count}.jpg")
                if process_standard(full_img_path, save_full_path): current_count += 1

# Sonuç Raporu
print("\nVERİ SETİ DAĞILIMI:")
for c in CLASSES:
    folder_path = os.path.join(MASTER_DIR, c)
    count = len(os.listdir(folder_path)) if os.path.exists(folder_path) else 0
    print(f"- {c.capitalize()}: {count} adet")

⚡ 'master_dataset' zaten dolu! Uzun süren arka plan silme işlemi atlanıyor...

📊 VERİ SETİ DAĞILIMI:
- Plastic: 1500 adet
- Glass: 1500 adet
- Paper: 1500 adet
- Metal: 1500 adet
- Trash: 1433 adet
- Cardboard: 1500 adet


### Adım 4: Data Augmentation ve Koruma Kalkanı

Overfitting'i önlemek için fotoğraflara rastgele döndürme, kaydırma ve yakınlaştırma işlemleri uygulanmıştır.

Bir fotoğraf kod ile döndürüldüğünde kenarlarda oluşan boşluklar varsayılan olarak piksellerin uzatılmasıyla doldurulur. Bu durum Siyah Hazne konseptini bozacağı için, fill_mode='constant' ve cval=0 parametreleri sisteme zorunlu koşularak rotasyon boşluklarının simsiyah kalması garanti altına alınmıştır.

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
    fill_mode='constant',
    cval=0
)

train_generator = datagen.flow_from_directory(
    MASTER_DIR, target_size=(224, 224), batch_size=32,
    class_mode='categorical', subset='training'
)

val_generator = datagen.flow_from_directory(
    MASTER_DIR, target_size=(224, 224), batch_size=32,
    class_mode='categorical', subset='validation'
)
class_names = list(train_generator.class_indices.keys())

Found 7147 images belonging to 6 classes.
Found 1786 images belonging to 6 classes.


### Adım 5: Model Mimarisi (MobileNetV2) ve Derleme

Projede hafif yapısı ve yüksek hızı sebebiyle Streamlit web arayüzünde canlı çalışmaya en uygun Transfer Learning mimarisi olan MobileNetV2 tercih edilmiştir.

Önceden eğitilmiş ImageNet ağırlıkları kullanılarak özellik çıkarımı hızlandırılmış; mimarinin sonuna eklenen %50 Dropout katmanı sayesinde modelin eğitim sırasında verileri ezberlemesinin önüne geçilmiştir.

In [10]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # İlk aşamada donduruyoruz (1.Aşama Eğitim için)

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5), # Aşırı öğrenmeyi engelle
    Dense(6, activation='softmax') # 6 Çöp Sınıfı Çıkışı
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

### Adım 6: 1. Aşama Eğitim

Bu aşamada sadece kendi eklediğimiz Dense katmanını eğitiyoruz. Olası bir ezberleme durumuna karşı önlem olarak EarlyStopping mekanizması eklenmiş ve sadece en yüksek validation başarısına ulaşan anın .keras formatında kaydedilmesi için ModelCheckpoint kullanılmıştır.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("akilli_kutu_model.keras", monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

print("🚀 1. Aşama Eğitim Başlıyor...")
history = model.fit(
    train_generator, epochs=50,
    validation_data=val_generator,
    callbacks=[early_stop, checkpoint]
)

🚀 1. Aşama Eğitim Başlıyor...
Epoch 1/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step - accuracy: 0.4099 - loss: 1.6135
Epoch 1: val_accuracy improved from None to 0.65957, saving model to akilli_kutu_model.keras

Epoch 1: finished saving model to akilli_kutu_model.keras
224/224 ━━━━━━━━━━━━━━━━━━━━ 125s 479ms/step - accuracy: 0.5057 - loss: 1.3558 - val_accuracy: 0.6596 - val_loss: 0.9551
Epoch 2/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.6255 - loss: 1.0343
Epoch 2: val_accuracy improved from 0.65957 to 0.69541, saving model to akilli_kutu_model.keras

Epoch 2: finished saving model to akilli_kutu_model.keras
224/224 ━━━━━━━━━━━━━━━━━━━━ 82s 364ms/step - accuracy: 0.6292 - loss: 1.0083 - val_accuracy: 0.6954 - val_loss: 0.8696
Epoch 3/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step - accuracy: 0.6683 - loss: 0.8887
Epoch 3: val_accuracy improved from 0.69541 to 0.71837, saving model to akilli_kutu_model.keras

Epoch 3: finished saving model to akilli_kutu_model.keras
224

### Adım 7: Fine-Tuning ile Maksimum Performans

Başlangıçtaki accuracy edefimiz olan %90+ başarı oranına ulaşmak için dondurulmuş MobileNetV2 mimarisinin son 55 katmanının kilidi açılmıştır. Modelin önceden öğrendiği temel şekil ve renk hafızasını silmemek amacıyla öğrenme hızı 100 kat düşürülerek 1e-5 seviyesine çekilmiş ve modele hassas ayar yapılmıştır. Bu hamle başarı oranını %93+ bandına taşımıştır.

In [ ]:
# Base Modelin kilidini aç (Son 55 katmanı serbest bırak)
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

# Öğrenme hızını çok düşür (1e-5)
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

early_stop_fine = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
checkpoint_fine = ModelCheckpoint("akilli_kutu_model.keras", monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

print("🚀 Fine-Tuning (Hassas Ayar) başlıyor...")
history_fine = model.fit(
    train_generator,
    epochs=50 + 50, # Toplam tur
    initial_epoch=history.epoch[-1],
    validation_data=val_generator,
    callbacks=[early_stop_fine, checkpoint_fine]
)

### Adım 8: Final Değerlendirmesi ve Karmaşıklık Matrisi

Eğitim tamamlandıktan sonra, jüri savunması için sektör standardı olan Karmaşıklık Matrisi ve detaylı sınıflandırma raporu görselleştirilmiştir. Bu aşama için veriler shuffle=False parametresiyle sıralı olarak tahmin motoruna sokulmuştur.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_generator = datagen.flow_from_directory(
    MASTER_DIR, target_size=(224, 224), batch_size=32,
    class_mode='categorical', subset='validation', shuffle=False # Matris için sıralı olmalı
)

predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Final Karmaşıklık Matrisi', fontsize=14)
plt.ylabel('Gerçek Sınıflar')
plt.xlabel('Tahmin Edilen Sınıflar')
plt.xticks(rotation=45)
plt.show()

print("\n📊 FİNAL DETAYLI RAPOR:")
print(classification_report(y_true, y_pred, target_names=class_names))

### Adım 9: Model Deployment

Eğitimi tamamlanan ve testlerden %93 başarıyla geçen nihai modelimiz, Streamlit web arayüzünde kullanılmak üzere dışa aktarılmıştır.

In [ ]:
from google.colab import files
files.download('akilli_kutu_model.keras')